# Module 02: Pandas for Machine Learning
## Notebook 06: Feature Engineering and Dataset Preparation

Raw data must be converted into numerical matrices suitable for Scikit-Learn estimators. This final notebook in Module 02 covers categorical encoding, numeric discretization (binning), outlier handling, and train-validation partitioning.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Apply One-Hot Encoding with `pd.get_dummies` avoiding the Dummy Variable Trap.
2. Convert ordered categories into numeric ranks with Ordinal Mapping.
3. Discretize continuous variables into discrete buckets using `pd.cut` and `pd.qcut`.
4. Detect and cap outliers using the Interquartile Range (IQR) rule.
5. Partition tabular data into feature matrix $X$ and target vector $y$.
6. **Advanced:** Construct leak-free Out-of-Fold (OOF) Smoothed Target Encoding and engineer cyclical sine/cosine transformations for periodic temporal features.

In [1]:
import pandas as pd
import numpy as np

# Create an end-to-end dataset for a credit risk prediction task
np.random.seed(42)
n_rows = 10

raw_data = {
    'Applicant_Age': [22, 45, 33, 58, 29, 39, 52, 24, 61, 35],
    'Education': ['High School', 'Bachelor', 'Master', 'PhD', 'Bachelor', 'Master', 'Bachelor', 'High School', 'PhD', 'Bachelor'],
    'Home_Ownership': ['RENT', 'MORTGAGE', 'RENT', 'OWN', 'RENT', 'MORTGAGE', 'OWN', 'RENT', 'MORTGAGE', 'RENT'],
    'Annual_Income': [35000, 85000, 62000, 140000, 48000, 95000, 110000, 28000, 450000, 72000],  # 450000 is an outlier
    'Defaulted': [1, 0, 0, 0, 1, 0, 0, 1, 0, 0]
}

df_loan = pd.DataFrame(raw_data)
print("Raw Loan Application Dataset:")
print(df_loan)

Raw Loan Application Dataset:
   Applicant_Age    Education Home_Ownership  Annual_Income  Defaulted
0             22  High School           RENT          35000          1
1             45     Bachelor       MORTGAGE          85000          0
2             33       Master           RENT          62000          0
3             58          PhD            OWN         140000          0
4             29     Bachelor           RENT          48000          1
5             39       Master       MORTGAGE          95000          0
6             52     Bachelor            OWN         110000          0
7             24  High School           RENT          28000          1
8             61          PhD       MORTGAGE         450000          0
9             35     Bachelor           RENT          72000          0


---
### 1. Categorical Encoding: One-Hot Encoding vs. Ordinal Mapping

Algorithms require numeric inputs. Two primary encoding paradigms:
- **Nominal Categories (Unordered):** One-Hot Encoding creates binary indicator columns.
  - Setting `drop_first=True` drops the baseline category, preventing exact multicollinearity (the **Dummy Variable Trap**) in linear models.
- **Ordinal Categories (Inherent Rank):** Use `.map()` with an explicit integer ranking dictionary (e.g. High School $\to 1$, Bachelor $\to 2$, Master $\to 3$, PhD $\to 4$).

In [2]:
# A. Ordinal Mapping: Education has an inherent natural hierarchy
education_rank = {
    'High School': 1,
    'Bachelor': 2,
    'Master': 3,
    'PhD': 4
}
df_loan['Education_Rank'] = df_loan['Education'].map(education_rank)

# B. One-Hot Encoding: Home Ownership has no order
df_encoded = pd.get_dummies(df_loan, columns=['Home_Ownership'], drop_first=True, dtype=int)

print("DataFrame after Ordinal and One-Hot Encoding:")
print(df_encoded.drop(columns=['Education']))

DataFrame after Ordinal and One-Hot Encoding:
   Applicant_Age  Annual_Income  Defaulted  Education_Rank  \
0             22          35000          1               1   
1             45          85000          0               2   
2             33          62000          0               3   
3             58         140000          0               4   
4             29          48000          1               2   
5             39          95000          0               3   
6             52         110000          0               2   
7             24          28000          1               1   
8             61         450000          0               4   
9             35          72000          0               2   

   Home_Ownership_OWN  Home_Ownership_RENT  
0                   0                    1  
1                   0                    0  
2                   0                    1  
3                   1                    0  
4                   0                    1  
5

---
### 2. Numerical Discretization (Binning): `cut` vs. `qcut`

Converting continuous variables into bins can capture non-linear relationships:
- **`pd.cut` (Equal-Width):** Divides the range of the feature into $K$ intervals of equal width.
- **`pd.qcut` (Equal-Frequency):** Divides data into quantiles so that each bin contains an equal number of samples.

In [3]:
# Equal-width age binning
df_encoded['Age_Group_Cut'] = pd.cut(df_encoded['Applicant_Age'], bins=[18, 30, 45, 65], labels=['Young', 'Middle', 'Senior'])

# Equal-frequency income quantiles (Tertiaries: Low, Mid, High)
df_encoded['Income_Tier_Qcut'] = pd.qcut(df_encoded['Annual_Income'], q=3, labels=['Low_Income', 'Mid_Income', 'High_Income'])

print("Discretized Features:")
print(df_encoded[['Applicant_Age', 'Age_Group_Cut', 'Annual_Income', 'Income_Tier_Qcut']])

Discretized Features:
   Applicant_Age Age_Group_Cut  Annual_Income Income_Tier_Qcut
0             22         Young          35000       Low_Income
1             45        Middle          85000       Mid_Income
2             33        Middle          62000       Low_Income
3             58        Senior         140000      High_Income
4             29         Young          48000       Low_Income
5             39        Middle          95000       Mid_Income
6             52        Senior         110000      High_Income
7             24         Young          28000       Low_Income
8             61        Senior         450000      High_Income
9             35        Middle          72000       Mid_Income


---
### 3. Outlier Detection via Interquartile Range (IQR)

Extreme outliers can distort mean-based estimators like Linear Regression and Neural Networks.
The **Tukey Boxplot Rule**:
$$\text{IQR} = Q_3 - Q_1$$
$$\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}$$
$$\text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$$

In [4]:
Q1 = df_encoded['Annual_Income'].quantile(0.25)
Q3 = df_encoded['Annual_Income'].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print(f"Q1 (25th): ${Q1:,.2f} | Q3 (75th): ${Q3:,.2f} | IQR: ${IQR:,.2f}")
print(f"Acceptable Range: [${lower_limit:,.2f}, ${upper_limit:,.2f}]")

# Outlier clipping / winsorization
df_encoded['Annual_Income_Capped'] = df_encoded['Annual_Income'].clip(lower=lower_limit, upper=upper_limit)
print("\nIncome Before and After IQR Capping:")
print(df_encoded[['Annual_Income', 'Annual_Income_Capped']])

Q1 (25th): $51,500.00 | Q3 (75th): $106,250.00 | IQR: $54,750.00
Acceptable Range: [$-30,625.00, $188,375.00]

Income Before and After IQR Capping:
   Annual_Income  Annual_Income_Capped
0          35000                 35000
1          85000                 85000
2          62000                 62000
3         140000                140000
4          48000                 48000
5          95000                 95000
6         110000                110000
7          28000                 28000
8         450000                188375
9          72000                 72000


---
### 4. Partitioning into Feature Matrix $X$ and Target Vector $y$

Before handing data over to Scikit-Learn or PyTorch:
1. Separate target label $y$ from feature columns $X$.
2. Drop raw non-numeric columns and identifier tags.
3. Validate that $X$ contains only numeric types (`int`, `float`, `bool`).

In [5]:
# Target vector y
y = df_encoded['Defaulted'].values

# Feature matrix X (Drop target and intermediate text/bin columns)
cols_to_drop = ['Defaulted', 'Annual_Income', 'Education_Rank', 'Age_Group_Cut', 'Income_Tier_Qcut']
if 'Education' in df_encoded.columns:
    cols_to_drop.append('Education')

X_df = df_encoded.drop(columns=cols_to_drop)
X = X_df.values

print(f"Feature Matrix X Shape: {X.shape} (N={X.shape[0]} samples, D={X.shape[1]} features)")
print(f"Target Vector y Shape:   {y.shape}")
print("\nFeature Columns ready for Scikit-Learn:")
print(list(X_df.columns))

Feature Matrix X Shape: (10, 4) (N=10 samples, D=4 features)
Target Vector y Shape:   (10,)

Feature Columns ready for Scikit-Learn:
['Applicant_Age', 'Home_Ownership_OWN', 'Home_Ownership_RENT', 'Annual_Income_Capped']


---
### 5. Advanced Complex Usage: Leak-Free Out-of-Fold Smoothed Target Encoding & Cyclical Feature Engineering

In high-cardinality categorical features (e.g. zip codes, device IDs, merchants):
- One-hot encoding creates thousands of sparse columns, causing memory bloat and curse of dimensionality.
- **Target Encoding** replaces each category with the average target value for that category.
- **Critical Risk:** Naive target encoding causes massive target leakage and severe overfitting.
- **The Production Fix:** **Out-of-Fold (OOF) K-Fold Target Encoding with Bayesian Smoothing (m-estimate)**:
$$S_i = \frac{n \cdot \bar{y}_c + m \cdot \bar{y}_{\text{global}}}{n + m}$$
where $m$ is the smoothing weight and $n$ is category frequency.

Additionally, cyclical temporal features (hours of day $0 \dots 23$, months $1 \dots 12$) must be projected onto a circle using sine and cosine transformations so that $23:00$ and $00:00$ are recognized as mathematically adjacent!

In [6]:
# 1. Out-of-Fold Smoothed Target Encoding Implementation
from sklearn.model_selection import KFold

# Synthetic high-cardinality zip code dataset with default target
np.random.seed(42)
n_samples_te = 100
zip_codes = np.random.choice(['ZIP_10001', 'ZIP_90210', 'ZIP_60601', 'ZIP_30301'], size=n_samples_te)
target_prob = {'ZIP_10001': 0.1, 'ZIP_90210': 0.05, 'ZIP_60601': 0.35, 'ZIP_30301': 0.60}
targets = [np.random.binomial(1, target_prob[z]) for z in zip_codes]

df_te = pd.DataFrame({'Zip': zip_codes, 'Target': targets})

# Initialize out-of-fold column
df_te['Target_Encoded_OOF'] = np.nan
kf = KFold(n_splits=5, shuffle=True, random_state=42)
smoothing_weight = 10.0
global_mean = df_te['Target'].mean()

for train_idx, val_idx in kf.split(df_te):
    train_fold = df_te.iloc[train_idx]
    
    # Calculate fold stats (strictly from training fold!)
    stats = train_fold.groupby('Zip')['Target'].agg(['count', 'mean'])
    
    # Bayesian smoothed formula
    smoothed = (stats['count'] * stats['mean'] + smoothing_weight * global_mean) / (stats['count'] + smoothing_weight)
    
    # Map back to validation fold
    df_te.iloc[val_idx, df_te.columns.get_loc('Target_Encoded_OOF')] = df_te.iloc[val_idx]['Zip'].map(smoothed).fillna(global_mean)

print("Leak-Free Out-of-Fold Smoothed Target Encoding Sample (First 6 rows):")
print(df_te.head(6))

# 2. Cyclical Temporal Feature Engineering (Sine/Cosine Transformation)
time_df = pd.DataFrame({'Hour': [0, 6, 12, 18, 23]})

# 24-hour cycle
time_df['Hour_Sin'] = np.sin(2 * np.pi * time_df['Hour'] / 24).round(4)
time_df['Hour_Cos'] = np.cos(2 * np.pi * time_df['Hour'] / 24).round(4)

print("\nCyclical Sine & Cosine Hour Representation:")
print(time_df)

# Distance between Hour 23 and Hour 0 in Euclidean space:
p0 = time_df.loc[time_df['Hour'] == 0, ['Hour_Sin', 'Hour_Cos']].values[0]
p23 = time_df.loc[time_df['Hour'] == 23, ['Hour_Sin', 'Hour_Cos']].values[0]
dist_23_0 = np.linalg.norm(p0 - p23)

p12 = time_df.loc[time_df['Hour'] == 12, ['Hour_Sin', 'Hour_Cos']].values[0]
dist_12_0 = np.linalg.norm(p0 - p12)

print(f"\nEuclidean Distance between Hour 23 and Hour 0:  {dist_23_0:.4f} (Close!)")
print(f"Euclidean Distance between Hour 12 and Hour 0:  {dist_12_0:.4f} (Distant!)")

Leak-Free Out-of-Fold Smoothed Target Encoding Sample (First 6 rows):
         Zip  Target  Target_Encoded_OOF
0  ZIP_60601       1            0.278571
1  ZIP_30301       0            0.508571
2  ZIP_10001       1            0.165217
3  ZIP_60601       1            0.278571
4  ZIP_60601       0            0.278571
5  ZIP_30301       0            0.494118

Cyclical Sine & Cosine Hour Representation:
   Hour  Hour_Sin  Hour_Cos
0     0    0.0000    1.0000
1     6    1.0000    0.0000
2    12    0.0000   -1.0000
3    18   -1.0000   -0.0000
4    23   -0.2588    0.9659

Euclidean Distance between Hour 23 and Hour 0:  0.2610 (Close!)
Euclidean Distance between Hour 12 and Hour 0:  2.0000 (Distant!)


### Module 02 Conclusion & Congratulations!
In Module 02, you progressed from Series and DataFrame fundamentals to advanced machine learning feature preparation:
1. **01:** Series, DataFrames, and memory optimization with categorical types and MultiIndexes.
2. **02:** Label (`.loc`) vs integer (`.iloc`) slicing, compound boolean masking, and method chaining with `.pipe()`.
3. **03:** Missing data diagnostics, subgroup-conditional imputation, and named regex extraction.
4. **04:** Groupby split-apply-combine, custom `.agg`, `.transform`, pivot tables, and expanding cumulative metrics.
5. **05:** Relational merges, concatenation, datetime indexing, rolling windows, and `pd.merge_asof`.
6. **06:** One-hot encoding, ordinal mappings, IQR clipping, cyclical transformations, and out-of-fold smoothed target encoding.

**Next Module:** `03_matplotlib` — Visualization foundations, the Object-Oriented interface, statistical charts, and ML diagnostics.